# CUDA Programming Lab: Data-Parallel Algorithms with PyCUDA

**Objective:** Understand GPU parallelism through hands-on implementation and benchmarking of fundamental CUDA kernels.

| # | Topic | Kernels |
|---|-------|---------|
| 0 | **CUDA Introduction** | Thread indexing, vector add, race conditions, atomics |
| 1 | Parallel Reduction | Naive, Shared-Memory Tree |
| 2 | Parallel Prefix Sum | Hillis-Steele (inclusive), Blelloch (exclusive) |
| 3 | Vector Dot Product | Shared-mem tree + atomicAdd |
| 4 | Matrix-Vector Product | V1 Naive, V2 Tiled-x, V3 2D Parallel |
| 5 | SGEMM (Matrix×Matrix) | K1 Naive → K2 GMEM Coalesce → K3 SMEM Tiled → K4 1D Blocktile → K5 2D Blocktile → cuBLAS |

Each section loads the kernel from the `kernels/` directory, verifies correctness against NumPy, and benchmarks performance.

In [ ]:
!pip install numpy matplotlib pycuda scikit-cuda ipykernel


## Setup & GPU Info

In [ ]:
import numpy as np
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import pycuda.gpuarray as gpuarray
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

KERNEL_DIR = Path("kernels")

dev  = cuda.Device(0)
attr = dev.get_attributes()
print(f"GPU               : {dev.name()}")
print(f"Compute Capability: {dev.compute_capability()}")
print(f"Total DRAM        : {dev.total_memory() / 2**30:.2f} GiB")
print(f"SMs               : {attr[cuda.device_attribute.MULTIPROCESSOR_COUNT]}")
print(f"Max threads/block : {attr[cuda.device_attribute.MAX_THREADS_PER_BLOCK]}")
print(f"Shared mem/block  : {attr[cuda.device_attribute.MAX_SHARED_MEMORY_PER_BLOCK] // 1024} KiB")
print(f"Warp size         : {attr[cuda.device_attribute.WARP_SIZE]}")

In [ ]:
def load_kernel(filename, options=None):
    """Compile a .cu file from kernels/ and return a SourceModule."""
    src = (KERNEL_DIR / filename).read_text()
    kw  = {"options": options} if options else {}
    return SourceModule(src, **kw)

def benchmark_gpu(func, warmup=5, repeats=50):
    """CUDA-event-based benchmark. Returns mean elapsed time in seconds."""
    start, end = cuda.Event(), cuda.Event()
    for _ in range(warmup): func()
    cuda.Context.synchronize()
    start.record()
    for _ in range(repeats): func()
    end.record(); end.synchronize()
    return start.time_till(end) * 1e-3 / repeats  # seconds

print("Utilities ready ✓")

---
## 0. Introduction to CUDA Programming

### 0.1 Why GPUs? The Parallelism Argument

A modern CPU has ~8–64 **powerful** cores optimised for low-latency serial execution.  
A modern GPU has **thousands of simpler cores** designed for high-throughput data-parallel work.

```
CPU:  [Core 0] [Core 1] ... [Core 63]        ← few, fast, flexible
GPU:  [SM 0] [SM 1] ... [SM 109]             ← many SMs, each running 100s of threads
```

**Rule of thumb:** If you can say _"do the same operation on many independent data elements"_, a GPU wins by 10×–100×.

---

### 0.2 The CUDA Execution Hierarchy

```
Grid  (entire kernel launch)
  └── Block 0 │ Block 1 │ Block 2 │ ...     (blockIdx.x/y/z)
        └── Thread 0 │ Thread 1 │ ...       (threadIdx.x/y/z)
```

| Variable | Meaning |
|----------|---------|
| `threadIdx.x` | Thread position within its block |
| `blockDim.x`  | Number of threads per block |
| `blockIdx.x`  | Block position within the grid |
| `gridDim.x`   | Number of blocks in the grid |

**Global thread index (1D):**
```cpp
int idx = blockIdx.x * blockDim.x + threadIdx.x;
```

Max 1024 threads per block. A **warp** = 32 threads executing in lockstep — the hardware's true unit of parallelism.

---

### 0.3 Kernel Syntax

```cpp
__global__ void my_kernel(float *data, int n) {   // runs on GPU
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) { data[idx] *= 2.0f; }
}

// Launched from CPU:
int block_size = 256;
int grid_size  = (n + block_size - 1) / block_size;   // ceiling division
my_kernel<<<grid_size, block_size>>>(d_data, n);
```

---

### 0.4 GPU Memory Hierarchy

| Memory | Scope | Speed | Keyword |
|--------|-------|-------|---------|
| Register | Per thread | ~30 TB/s | local variables |
| Shared Memory | Per block | ~10 TB/s | `__shared__` |
| L2 Cache | Chip-wide | ~3 TB/s | automatic |
| Global Memory (DRAM) | All threads | ~0.5–2 TB/s | `cudaMalloc` |

**Golden rule:** Keep data as close to compute as possible. DRAM is the bottleneck.

**Host ↔ Device flow (in PyCUDA terms):**
```python
d_a = gpuarray.to_gpu(h_a)    # cudaMalloc + cudaMemcpy H→D
# ... launch kernel ...
h_a = d_a.get()               # cudaMemcpy D→H
```

### 0.5 Thread Indexing Demo

Every thread computes its unique global ID and writes it to an array. This makes the `blockIdx * blockDim + threadIdx` formula concrete.

In [ ]:
mod_intro = load_kernel("intro.cu")
thread_index_demo = mod_intro.get_function("thread_index_demo")

N_idx   = 32  # small so we can inspect every element
BLOCK   = 8
GRID    = N_idx // BLOCK
d_out   = gpuarray.zeros(N_idx, dtype=np.int32)

thread_index_demo(d_out, np.int32(N_idx), block=(BLOCK,1,1), grid=(GRID,1,1))
cuda.Context.synchronize()

result = d_out.get()
print(f"Launching {GRID} blocks × {BLOCK} threads = {GRID*BLOCK} total threads")
print(f"Output array (each element = thread's global ID):")
print(result.reshape(GRID, BLOCK))
print()
print("Expected: 0, 1, 2, ...", N_idx - 1)
print("Correct ✓" if np.all(result == np.arange(N_idx)) else "Error ✗")

### 0.6 Hello GPU: Vector Addition

The canonical first CUDA kernel.  
- CPU: loops over N elements sequentially — O(N) time  
- GPU: N threads each add one element — O(1) depth

```cpp
__global__ void vector_add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];   // one thread, one element
}
```

In [ ]:
vector_add_gpu = mod_intro.get_function("vector_add")

N_va = 10_000_000
h_a  = np.random.rand(N_va).astype(np.float32)
h_b  = np.random.rand(N_va).astype(np.float32)
d_a, d_b = gpuarray.to_gpu(h_a), gpuarray.to_gpu(h_b)
d_c  = gpuarray.zeros_like(d_a)

BLOCK = 256
GRID  = int(np.ceil(N_va / BLOCK))
vector_add_gpu(d_a, d_b, d_c, np.int32(N_va), block=(BLOCK,1,1), grid=(GRID,1,1))
cuda.Context.synchronize()

# Correctness
cpu_ref = h_a + h_b
max_err = np.max(np.abs(d_c.get() - cpu_ref))
print(f"N = {N_va:,}  |  max element error = {max_err:.2e}  {'✓' if max_err < 1e-5 else '✗'}")

# CPU benchmark
import time
t0 = time.perf_counter()
for _ in range(5): _ = h_a + h_b
t_cpu = (time.perf_counter() - t0) / 5 * 1e3

# GPU benchmark
t_gpu = benchmark_gpu(
    lambda: vector_add_gpu(d_a, d_b, d_c, np.int32(N_va), block=(BLOCK,1,1), grid=(GRID,1,1))
) * 1e3

bw_gpu = 3 * N_va * 4 / (t_gpu/1e3) / 1e9  # read a, read b, write c
print(f"CPU add : {t_cpu:.2f} ms")
print(f"GPU add : {t_gpu:.2f} ms  |  {bw_gpu:.1f} GB/s  |  {t_cpu/t_gpu:.1f}× faster dan CPU")

### 0.7 Race Conditions & Atomic Operations

What happens when **1,000,000 threads all increment the same counter**?

```
Thread 0: read counter=5, compute 5+1=6, write 6     ─┐ Both read 5!
Thread 1: read counter=5, compute 5+1=6, write 6     ─┘ One increment is lost.
```

`atomicAdd` wraps the read-modify-write into a single **indivisible** hardware instruction — no two threads can interleave.

In [ ]:
counter_race   = mod_intro.get_function("counter_race")
counter_atomic = mod_intro.get_function("counter_atomic")

NUM_BLOCKS   = 1000
NUM_THREADS  = 1000
EXPECTED     = NUM_BLOCKS * NUM_THREADS  # 1,000,000

# --- Broken version (race condition) ---
d_race = gpuarray.zeros(1, dtype=np.int32)
counter_race(d_race, block=(NUM_THREADS,1,1), grid=(NUM_BLOCKS,1,1))
cuda.Context.synchronize()
race_result = int(d_race.get()[0])

# --- Fixed version (atomicAdd) ---
d_atomic = gpuarray.zeros(1, dtype=np.int32)
counter_atomic(d_atomic, block=(NUM_THREADS,1,1), grid=(NUM_BLOCKS,1,1))
cuda.Context.synchronize()
atomic_result = int(d_atomic.get()[0])

print(f"Expected              : {EXPECTED:,}")
print(f"Race (broken)  result : {race_result:,}  ← {EXPECTED - race_result:,} increments lost!")
print(f"Atomic (fixed) result : {atomic_result:,}  ✓" if atomic_result == EXPECTED else f"Atomic result: {atomic_result:,}  ✗")

**Key takeaway:** Use `atomicAdd` when threads must update shared state. But note it **serialises** access — heavy contention on a single address becomes a bottleneck. The fix: reduce within a block first (tree reduction), then `atomicAdd` once per block.

---

### 0.8 PyCUDA Quick Reference

| PyCUDA | C CUDA equivalent |
|--------|-------------------|
| `gpuarray.to_gpu(arr)` | `cudaMalloc` + `cudaMemcpy(H→D)` |
| `gpu_arr.get()` | `cudaMemcpy(D→H)` |
| `SourceModule(src_string)` | `nvcc` compilation |
| `mod.get_function("name")` | kernel function pointer |
| `fn(..., block=(Bx,By,Bz), grid=(Gx,Gy,Gz))` | `fn<<<grid,block>>>(...)` |
| `cuda.Context.synchronize()` | `cudaDeviceSynchronize()` |

---
## 1. Parallel Reduction

**Goal:** Sum N floats in parallel using a tree — depth O(log N) instead of O(N).

```
Step 1  [x0+x1] [x2+x3] [x4+x5] [x6+x7]   stride = blockDim/2
Step 2  [x0+..+x3]      [x4+..+x7]          stride = blockDim/4
Step 3  [x0+..+x7]                           stride = 1
```

| Kernel | Strategy | Depth | Work |
|--------|----------|-------|------|
| Naive | Thread 0 sums whole block sequentially | O(N) | O(N) |
| Shared-Mem Tree | Halve active threads each step | O(log N) | O(N) |

In [ ]:
mod_red = load_kernel("reduction.cu")
naive_reduce   = mod_red.get_function("naive_reduction")
tree_reduce    = mod_red.get_function("shared_mem_reduction")

def two_pass_reduce(d_data, kernel, block=256):
    """Block-level reduce → CPU sum of partial results."""
    n      = d_data.size
    grid   = (int(np.ceil(n / block)), 1, 1)
    partials = gpuarray.zeros(grid[0], dtype=np.float32)
    kernel(d_data, partials, np.int32(n),
           block=(block, 1, 1), grid=grid, shared=block * 4)
    cuda.Context.synchronize()
    return float(partials.get().sum())

N = 1 << 20
h  = np.random.rand(N).astype(np.float32)
d  = gpuarray.to_gpu(h)

cpu_s  = h.sum()
nv_s   = two_pass_reduce(d, naive_reduce)
tr_s   = two_pass_reduce(d, tree_reduce)

print(f"N = {N:,}")
print(f"CPU (numpy)      = {cpu_s:.4f}")
print(f"GPU naive        = {nv_s:.4f}  err={abs(nv_s-cpu_s)/cpu_s*100:.4f}%")
print(f"GPU shared tree  = {tr_s:.4f}  err={abs(tr_s-cpu_s)/cpu_s*100:.4f}%")

In [ ]:
BLOCK, GRID = 256, int(np.ceil(N / 256))
dummy = gpuarray.zeros(GRID, dtype=np.float32)

t_nv = benchmark_gpu(lambda: naive_reduce(d, dummy, np.int32(N),
    block=(BLOCK,1,1), grid=(GRID,1,1), shared=BLOCK*4))
t_tr = benchmark_gpu(lambda: tree_reduce(d, dummy, np.int32(N),
    block=(BLOCK,1,1), grid=(GRID,1,1), shared=BLOCK*4))

bw_nv = N*4 / t_nv / 1e9
bw_tr = N*4 / t_tr / 1e9
print(f"Naive  : {t_nv*1e3:.3f} ms  {bw_nv:.1f} GB/s")
print(f"Tree   : {t_tr*1e3:.3f} ms  {bw_tr:.1f} GB/s  [{t_nv/t_tr:.2f}× faster]")

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['Naive', 'Shared-Mem\nTree'], [bw_nv, bw_tr], color=['#e76f51','#2a9d8f'], width=0.4)
ax.set_ylabel('Effective Bandwidth (GB/s)')
ax.set_title(f'Parallel Reduction  (N={N//1<<20}M floats)')
plt.tight_layout(); plt.show()

---
## 2. Parallel Prefix Sum (Scan)

**Goal:** `out[i] = sum(in[0..i-1])` for every i in parallel.

```
Input:          [ 1,  2,  3,  4 ]
Inclusive scan: [ 1,  3,  6, 10 ]   (Hillis-Steele)
Exclusive scan: [ 0,  1,  3,  6 ]   (Blelloch)
```

| Algorithm | Depth | Work | Type |
|-----------|-------|------|------|
| Hillis-Steele | O(log N) | O(N log N) | Inclusive |
| Blelloch (up+down sweep) | O(log N) | O(N) | Exclusive |

In [ ]:
mod_scan = load_kernel("prefix_sum.cu")
hs_scan  = mod_scan.get_function("hillis_steele_scan")
bl_scan  = mod_scan.get_function("blelloch_scan")

N_s  = 1024
h_in = np.random.randint(1, 5, N_s).astype(np.float32)
d_in = gpuarray.to_gpu(h_in)

cpu_inc = np.cumsum(h_in)
cpu_exc = np.concatenate([[0], cpu_inc[:-1]])

# Hillis-Steele (inclusive)
d_hs = gpuarray.zeros(N_s, dtype=np.float32)
hs_scan(d_in, d_hs, np.int32(N_s),
        block=(N_s,1,1), grid=(1,1,1), shared=N_s*4)
cuda.Context.synchronize()

# Blelloch (exclusive, block = N_s//2 because kernel processes pairs)
d_bl = gpuarray.zeros(N_s, dtype=np.float32)
bl_scan(d_in, d_bl, np.int32(N_s),
        block=(N_s//2,1,1), grid=(1,1,1), shared=N_s*4)
cuda.Context.synchronize()

print("Input (first 8)    :", h_in[:8].astype(int).tolist())
print("Hillis-Steele out  :", d_hs.get()[:8].astype(int).tolist())
print("CPU inclusive      :", cpu_inc[:8].astype(int).tolist())
print("Blelloch out       :", d_bl.get()[:8].astype(int).tolist())
print("CPU exclusive      :", cpu_exc[:8].astype(int).tolist())
print()
print("Hillis-Steele ✓" if np.allclose(d_hs.get(), cpu_inc, atol=1) else "Hillis-Steele ✗")
print("Blelloch      ✓" if np.allclose(d_bl.get(), cpu_exc, atol=1) else "Blelloch      ✗")

In [ ]:
sizes_scan = [128, 256, 512, 1024]
t_hs_list, t_bl_list = [], []
for sz in sizes_scan:
    arr = gpuarray.to_gpu(np.random.rand(sz).astype(np.float32))
    out = gpuarray.zeros(sz, dtype=np.float32)
    t_hs_list.append(benchmark_gpu(
        lambda: hs_scan(arr, out, np.int32(sz), block=(sz,1,1), grid=(1,1,1), shared=sz*4),
        warmup=3, repeats=200) * 1e6)
    t_bl_list.append(benchmark_gpu(
        lambda: bl_scan(arr, out, np.int32(sz), block=(sz//2,1,1), grid=(1,1,1), shared=sz*4),
        warmup=3, repeats=200) * 1e6)

x = np.arange(len(sizes_scan)); w = 0.35
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x-w/2, t_hs_list, w, label='Hillis-Steele', color='#e76f51')
ax.bar(x+w/2, t_bl_list, w, label='Blelloch',      color='#2a9d8f')
ax.set_xticks(x); ax.set_xticklabels(sizes_scan)
ax.set_xlabel('Array size'); ax.set_ylabel('Time (µs)')
ax.set_title('Prefix Scan: Hillis-Steele vs Blelloch')
ax.legend(); plt.tight_layout(); plt.show()

---
## 3. Vector Dot Product

**Goal:** `result = Σ A[i] * B[i]`  — element-wise multiply then reduce.

Each block handles a chunk, tree-reduces in shared memory, then `atomicAdd` to a single global accumulator.

**This kernel is memory-bound:** Arithmetic intensity = 2 FLOP / 8 bytes ≈ 0.25 FLOP/byte.

In [ ]:
mod_dot = load_kernel("dot_product.cu")
dot_gpu = mod_dot.get_function("dot_product")

N_dot = 1 << 22
hA, hB = np.random.rand(N_dot).astype(np.float32), np.random.rand(N_dot).astype(np.float32)
dA, dB = gpuarray.to_gpu(hA), gpuarray.to_gpu(hB)

BLOCK = 256
GRID  = int(np.ceil(N_dot / BLOCK))
d_res = gpuarray.zeros(1, dtype=np.float32)
dot_gpu(dA, dB, d_res, np.int32(N_dot), block=(BLOCK,1,1), grid=(GRID,1,1), shared=BLOCK*4)
cuda.Context.synchronize()

gpu_dot = float(d_res.get()[0])
cpu_dot = float(np.dot(hA.astype(np.float64), hB.astype(np.float64)))
print(f"N = {N_dot:,}")
print(f"GPU dot product = {gpu_dot:.4f}")
print(f"CPU dot product = {cpu_dot:.4f}  (float64 ref)")
print(f"Relative error  = {abs(gpu_dot-cpu_dot)/cpu_dot*100:.4f}%")

t_dot = benchmark_gpu(lambda: dot_gpu(dA, dB, d_res, np.int32(N_dot),
    block=(BLOCK,1,1), grid=(GRID,1,1), shared=BLOCK*4))
bw_dot = 2*N_dot*4 / t_dot / 1e9
print(f"\nTime: {t_dot*1e3:.3f} ms  |  Bandwidth: {bw_dot:.1f} GB/s")

---
## 4. Matrix-Vector Product  (y = A · x)

**Three progressively optimised versions:**

| Version | x loaded from GMEM | Parallelism per row |
|---------|-------------------|---------------------|
| V1 Naive | M × per thread | 1 thread |
| V2 Tiled-x | Once per block per tile | 1 thread |
| V3 2D Parallel | Once per block per tile | All BLOCK_COLS threads |

In [ ]:
TILE = 256
BCOLS = 256  # BLOCK_COLS for V3 — must be power of 2
mod_mv = load_kernel("matvec.cu", options=[f"-DTILE_SIZE={TILE}", f"-DBLOCK_COLS={BCOLS}"])
mv_naive    = mod_mv.get_function("matvec_naive")
mv_tiled    = mod_mv.get_function("matvec_tiled")
mv_parallel = mod_mv.get_function("matvec_parallel_rows")

M_mv, N_mv = 4096, 4096
hA_mv = np.random.rand(M_mv, N_mv).astype(np.float32)
hx_mv = np.random.rand(N_mv).astype(np.float32)
dA_mv = gpuarray.to_gpu(hA_mv)
dx_mv = gpuarray.to_gpu(hx_mv)
cpu_y = hA_mv @ hx_mv

BLOCK_MV = 256
GRID_MV  = int(np.ceil(M_mv / BLOCK_MV))

for name, fn, blk, grd in [
    ("V1 Naive",      mv_naive,    (BLOCK_MV,1,1), (GRID_MV,1,1)),
    ("V2 Tiled-x",    mv_tiled,    (BLOCK_MV,1,1), (GRID_MV,1,1)),
    ("V3 2D Parallel",mv_parallel, (BCOLS,1,1),    (M_mv,1,1)),
]:
    dy = gpuarray.zeros(M_mv, dtype=np.float32)
    fn(dA_mv, dx_mv, dy, np.int32(M_mv), np.int32(N_mv), block=blk, grid=grd)
    cuda.Context.synchronize()
    err = np.max(np.abs(dy.get() - cpu_y))
    print(f"{name:20s}: max_err={err:.2e}  {'✓' if err < 1e-2 else '✗'}")

In [ ]:
dy = gpuarray.zeros(M_mv, dtype=np.float32)
results_mv = {}
for name, fn, blk, grd in [
    ("V1 Naive",      mv_naive,    (BLOCK_MV,1,1), (GRID_MV,1,1)),
    ("V2 Tiled-x",    mv_tiled,    (BLOCK_MV,1,1), (GRID_MV,1,1)),
    ("V3 2D Parallel",mv_parallel, (BCOLS,1,1),    (M_mv,1,1)),
]:
    t = benchmark_gpu(lambda: fn(dA_mv, dx_mv, dy, np.int32(M_mv), np.int32(N_mv),
                                 block=blk, grid=grd))
    bw = (M_mv*N_mv + N_mv + M_mv)*4 / t / 1e9
    gf = 2*M_mv*N_mv / t / 1e9
    results_mv[name] = (t*1e3, bw, gf)
    print(f"{name:20s}: {t*1e3:.3f} ms  {bw:.1f} GB/s  {gf:.1f} GFLOPS")

fig, ax = plt.subplots(figsize=(7,4))
names = list(results_mv.keys())
gflops = [v[2] for v in results_mv.values()]
colors = ['#e76f51','#f4a261','#2a9d8f']
ax.bar(names, gflops, color=colors)
ax.set_ylabel('GFLOPS'); ax.set_title(f'Matrix-Vector Product  ({M_mv}×{N_mv})')
plt.tight_layout(); plt.show()

---
## 5. SGEMM  (C = α·A·B + β·C)  — 5 Kernels vs cuBLAS

### Optimisation Ladder

| Kernel | Key Idea | Bottleneck |
|--------|----------|------------|
| K1 Naive | 1 thread = 1 element | Uncoalesced GMEM reads |
| K2 GMEM Coalesce | Remap threads → coalesced warp accesses | GMEM bandwidth |
| K3 SMEM Tiled | Load tiles into shared memory — data reuse | SMEM bandwidth |
| K4 1D Blocktile | Each thread computes TM=8 elements → register reuse of B | SMEM loads |
| K5 2D Blocktile | Each thread computes TM×TN=64 elements → outer product in registers | Compute-bound! |
| cuBLAS | Vendor-tuned (warp-tiling, vectorization, double-buffering) | Hardware peak |

> **Memory Hierarchy Bandwidths (approximate):**
> ```
> Registers  ~30 TB/s  ← outer product lives here in K5
> SMEM       ~10 TB/s  ← tiles live here in K3/K4
> L2         ~ 3 TB/s
> DRAM       ~0.5 TB/s ← bottleneck in K1/K2
> ```

In [ ]:
TILE_S = 16
BM, BN, BK_val, TM_val = 64, 64, 8, 8
TN_val = 8

opts = [
    f"-DTILE_SIZE={TILE_S}",
    f"-DBM={BM}", f"-DBN={BN}", f"-DBK={BK_val}",
    f"-DTM={TM_val}", f"-DTN={TN_val}"
]
mod_mm = load_kernel("sgemm.cu", options=opts)

k1_naive     = mod_mm.get_function("sgemm_naive")
k2_coalesce  = mod_mm.get_function("sgemm_gmem_coalesce")
k3_smem      = mod_mm.get_function("sgemm_smem_tiled")
k4_1d        = mod_mm.get_function("sgemm_1d_blocktiling")
k5_2d        = mod_mm.get_function("sgemm_2d_blocktiling")

print(f"All 5 SGEMM kernels compiled ✓  (TILE={TILE_S}, BM/BN={BM}/{BN}, BK={BK_val}, TM/TN={TM_val}/{TN_val})")

In [ ]:
try:
    import skcuda.cublas as cublas
    HAS_CUBLAS = True
    print("scikit-cuda / cuBLAS available ✓")
except ImportError:
    HAS_CUBLAS = False
    print("scikit-cuda not found — skipping cuBLAS.\n  Install: pip install scikit-cuda")

In [ ]:
def run_k1(M, N, K, a, dA, dB, b, dC):
    BS = 32
    k1_naive(np.int32(M), np.int32(N), np.int32(K), np.float32(a), dA, dB, np.float32(b), dC,
             block=(BS,BS,1), grid=(int(np.ceil(M/BS)), int(np.ceil(N/BS)), 1))

def run_k2(M, N, K, a, dA, dB, b, dC):
    BS = 32
    k2_coalesce(np.int32(M), np.int32(N), np.int32(K), np.float32(a), dA, dB, np.float32(b), dC,
                np.int32(BS), block=(BS*BS,1,1), grid=(int(np.ceil(M/BS)), int(np.ceil(N/BS)), 1))

def run_k3(M, N, K, a, dA, dB, b, dC):
    k3_smem(np.int32(M), np.int32(N), np.int32(K), np.float32(a), dA, dB, np.float32(b), dC,
            block=(TILE_S*TILE_S,1,1), grid=(int(np.ceil(M/TILE_S)), int(np.ceil(N/TILE_S)), 1))

def run_k4(M, N, K, a, dA, dB, b, dC):
    k4_1d(np.int32(M), np.int32(N), np.int32(K), np.float32(a), dA, dB, np.float32(b), dC,
          block=(BM // TM_val * BN, 1, 1), grid=(int(np.ceil(N/BN)), int(np.ceil(M/BM)), 1))

def run_k5(M, N, K, a, dA, dB, b, dC):
    BM2, BN2 = 128, 128
    threads = (BM2 // TM_val) * (BN2 // TN_val)
    k5_2d(np.int32(M), np.int32(N), np.int32(K), np.float32(a), dA, dB, np.float32(b), dC,
          block=(threads,1,1), grid=(int(np.ceil(N/BN2)), int(np.ceil(M/BM2)), 1))

# Correctness check
Mc = Nc = Kc = 256
hAc = np.random.rand(Mc, Kc).astype(np.float32)
hBc = np.random.rand(Kc, Nc).astype(np.float32)
ref = hAc @ hBc

for kname, kfn in [("K1 Naive",   run_k1), ("K2 Coalesce", run_k2),
                   ("K3 SMEM",    run_k3), ("K4 1D-Tile",  run_k4),
                   ("K5 2D-Tile", run_k5)]:
    dA_c = gpuarray.to_gpu(hAc); dB_c = gpuarray.to_gpu(hBc)
    dC_c = gpuarray.zeros((Mc, Nc), dtype=np.float32)
    kfn(Mc, Nc, Kc, 1.0, dA_c, dB_c, 0.0, dC_c)
    cuda.Context.synchronize()
    err = np.max(np.abs(dC_c.get() - ref))
    print(f"{kname:12s}: max_err={err:.4f}  {'✓' if err < 0.5 else '✗'}")

In [ ]:
sizes_mm = [128, 256, 512, 1024, 2048]
kernel_specs = [
    ("K1 Naive",    run_k1, '#e63946'),
    ("K2 Coalesce", run_k2, '#f4a261'),
    ("K3 SMEM",     run_k3, '#e9c46a'),
    ("K4 1D-Tile",  run_k4, '#2a9d8f'),
    ("K5 2D-Tile",  run_k5, '#264653'),
]

perf = {name: [] for name, _, _ in kernel_specs}
perf["cuBLAS"] = []

for sz in sizes_mm:
    M = N = K = sz
    hA_b = np.random.rand(M, K).astype(np.float32)
    hB_b = np.random.rand(K, N).astype(np.float32)
    dA_b = gpuarray.to_gpu(hA_b)
    dB_b = gpuarray.to_gpu(hB_b)
    flops = 2 * M * N * K
    row = f"sz={sz:4d}"

    for kname, kfn, _ in kernel_specs:
        dC_b = gpuarray.zeros((M,N), dtype=np.float32)
        try:
            t = benchmark_gpu(lambda: kfn(M,N,K, 1.,dA_b,dB_b, 0.,dC_b), warmup=3, repeats=20)
            gf = flops / t / 1e9
        except Exception:
            gf = float('nan')
        perf[kname].append(gf)
        row += f"  {kname}={gf:.1f}"

    if HAS_CUBLAS:
        dC_b = gpuarray.zeros((M,N), dtype=np.float32)
        hdl  = cublas.cublasCreate()
        t_cb = benchmark_gpu(
            lambda: cublas.cublasSgemm(hdl,'n','n', N,M,K,
                                       np.float32(1.),dB_b.gpudata,N,
                                       dA_b.gpudata,K,
                                       np.float32(0.),dC_b.gpudata,N),
            warmup=5, repeats=50)
        cublas.cublasDestroy(hdl)
        gf_cb = flops / t_cb / 1e9
        perf["cuBLAS"].append(gf_cb)
        row += f"  cuBLAS={gf_cb:.1f}"
    else:
        perf["cuBLAS"].append(None)

    print(row)

In [ ]:
all_specs = kernel_specs + [("cuBLAS", None, '#8338ec')]
markers   = {'K1 Naive':'o','K2 Coalesce':'s','K3 SMEM':'^','K4 1D-Tile':'D','K5 2D-Tile':'P','cuBLAS':'*'}

fig, ax = plt.subplots(figsize=(11, 6))
for kname, _, color in all_specs:
    vals = perf[kname]
    xs = [sizes_mm[i] for i, v in enumerate(vals) if v is not None and not (isinstance(v, float) and np.isnan(v))]
    ys = [v            for v in vals            if v is not None and not (isinstance(v, float) and np.isnan(v))]
    if ys:
        ax.plot(xs, ys, marker=markers[kname], linewidth=2.5,
                label=kname, color=color, markersize=8)

ax.set_xlabel('Matrix Size N  (N×N square matrix)')
ax.set_ylabel('Performance (GFLOPS)')
ax.set_title('SGEMM: 5 Custom Kernels vs cuBLAS', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(loc='upper left')
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.savefig('sgemm_benchmark.png', dpi=150)
plt.show()
print("Saved sgemm_benchmark.png")

In [ ]:
print("\n===== SGEMM Performance Summary (GFLOPS) =====")
col_names = [n for n,_,_ in all_specs]
header = f"{'Size':>6}" + "".join(f"  {n:>12}" for n in col_names)
print(header)
print('-' * len(header))
for i, sz in enumerate(sizes_mm):
    row = f"{sz:>6}"
    for kname in col_names:
        v = perf[kname][i]
        row += f"  {v:>12.1f}" if v is not None and not (isinstance(v,float) and np.isnan(v)) else f"  {'N/A':>12}"
    print(row)

---
## 6. Summary & Takeaways

| Topic | Key Insight |
|-------|-------------|
| CUDA Intro | Each thread has a unique ID; warps of 32 execute in lockstep |
| Reduction | Tree halves work per step → O(log N) depth |
| Prefix Scan | Blelloch is O(N) work vs O(N log N) for Hillis-Steele |
| Dot Product | Memory-bound; maximize bandwidth utilization |
| MatVec V1→V3 | Sharing x in SMEM, then parallelising across columns of each row |
| SGEMM K1→K5 | Each step reduces memory pressure by keeping data closer to compute |

### Arithmetic Intensity (SGEMM)
```
K1 Naive    ≈ 0.25 FLOP/byte   → memory-bound (GMEM)
K3 SMEM     ≈ TILE FLOP/byte   → memory-bound (SMEM)
K5 2D-Tile  ≈ TM·TN FLOP/byte → compute-bound (registers)
```

**Further reading:**  
- Simon Boehm — [How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance](https://siboehm.com/articles/22/CUDA-MMM)  
- NVIDIA — [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/)  
- Stanford CS149 — Parallel Computing lecture notes  
- PyCUDA docs — [documen.tician.de/pycuda](https://documen.tician.de/pycuda/)